Imports

In [1]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

Load data

In [2]:
df = pd.read_excel('chatgpt_style_reviews_dataset.xlsx')
print('Shape:', df.shape)
df.head()

Shape: (500, 12)


,date,title,review,rating,username,helpful_votes,review_length,platform,language,location,version,verified_purchase
0,########,Review title 1,"Not satisfied, many bugs and issues.",1,user1,80,6,Amazon,zh,Kenya,2.1.4,No
1,########,Review title 2,Amazing quality and user-friendly interface.,5,user2,180,5,Flipkart,zh,France,1.2.3,No
2,########,Review title 3,"Terrible experience, needs major improvements.",2,user3,154,5,Flipkart,pt,USA,1.2.3,No
3,########,Review title 4,Poor performance and not user-friendly.,1,user4,96,5,Amazon,es,Qatar,2.1.4,Yes
4,########,Review title 5,"Not satisfied, many bugs and issues.",2,user5,139,6,Website,ar,Kenya,2.1.4,No


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   date               500 non-null    object
 1   title              500 non-null    str   
 2   review             500 non-null    str   
 3   rating             500 non-null    int64 
 4   username           500 non-null    str   
 5   helpful_votes      500 non-null    int64 
 6   review_length      500 non-null    int64 
 7   platform           500 non-null    str   
 8   language           500 non-null    str   
 9   location           500 non-null    str   
 10  version            500 non-null    str   
 11  verified_purchase  500 non-null    str   
dtypes: int64(3), object(1), str(8)
memory usage: 92.0+ KB


In [4]:
df['review'].nunique()

15

In [5]:
df['review'].value_counts()

review
Highly satisfied, the app works exactly as expected.         50
Waste of time, does not meet expectations.                   43
Very reliable and worth using regularly.                     43
Terrible experience, needs major improvements.               41
Poor performance and not user-friendly.                      41
Great experience, smooth performance and useful features.    39
Very disappointing experience, the app crashes often.        38
Excellent app, very easy to use and extremely helpful.       35
Amazing quality and user-friendly interface.                 33
Not satisfied, many bugs and issues.                         32
Not bad, but updates are needed for better performance.      24
Works fine but there is room for improvement.                23
The app is okay and does its job reasonably well.            22
Average experience, some features could be improved.         21
Decent app, neither too good nor too bad.                    15
Name: count, dtype: int64

date clean

In [6]:
df['date_clean'] = pd.to_datetime(df['date'], errors='coerce')
df[['date', 'date_clean']]

C:\Users\yogap\AppData\Local\Temp\ipykernel_14260\1311598116.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date_clean'] = pd.to_datetime(df['date'], errors='coerce')


,date,date_clean
0,########,NaT
1,########,NaT
2,########,NaT
3,########,NaT
4,########,NaT
...,...,...
495,2024-07-05 00:00:00,2024-07-05
496,########,NaT
497,########,NaT
498,########,NaT


In [7]:
df['date_clean'].isnull().sum()

np.int64(358)

clean review length

In [8]:
df['review_length_clean'] = df['review'].str.len()
df[['review', 'review_length','review_length_clean']]

,review,review_length,review_length_clean
0,"Not satisfied, many bugs and issues.",6,36
1,Amazing quality and user-friendly interface.,5,44
2,"Terrible experience, needs major improvements.",5,46
3,Poor performance and not user-friendly.,5,39
4,"Not satisfied, many bugs and issues.",6,36
...,...,...,...
495,"Waste of time, does not meet expectations.",7,42
496,"Great experience, smooth performance and usefu...",7,57
497,"Terrible experience, needs major improvements.",5,46
498,"Highly satisfied, the app works exactly as exp...",8,52


Tokenize & Lemmatize

In [9]:
stop_words = set(stopwords.words('english'))

negation_words = {"not", "no", "nor", "never", "cannot", "n't", "don", "didn", "doesn",
                   "isn", "wasn", "weren", "won", "wouldn", "couldn", "shouldn", "aren"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)      # urls
    text = re.sub(r'[^a-z\s]', ' ', text)                # punctuation / digits / special chars
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]   # verb forms
    tokens = [lemmatizer.lemmatize(t, pos='n') for t in tokens]   # noun forms
    return ' '.join(tokens)

df['review_clean'] = df['review'].apply(clean_text)
df['review_clean_tokens'] = df['review_clean'].apply(lambda x: x.split())
df['clean_token_count'] = df['review_clean_tokens'].apply(len)

df[['review', 'review_clean']]

,review,review_clean
0,"Not satisfied, many bugs and issues.",not satisfy many bug issue
1,Amazing quality and user-friendly interface.,amaze quality user friendly interface
2,"Terrible experience, needs major improvements.",terrible experience need major improvement
3,Poor performance and not user-friendly.,poor performance not user friendly
4,"Not satisfied, many bugs and issues.",not satisfy many bug issue
...,...,...
495,"Waste of time, does not meet expectations.",waste time not meet expectation
496,"Great experience, smooth performance and usefu...",great experience smooth performance useful fea...
497,"Terrible experience, needs major improvements.",terrible experience need major improvement
498,"Highly satisfied, the app works exactly as exp...",highly satisfy app work exactly expect


Sentiment classification

In [10]:
def rating_to_sentiment(r):
    if r <= 2:
        return 'Negative'
    elif r == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['rating'].apply(rating_to_sentiment)
df['sentiment'].value_counts()

sentiment
Positive    200
Negative    195
Neutral     105
Name: count, dtype: int64

In [12]:
df

,date,title,review,rating,username,helpful_votes,review_length,platform,language,location,version,verified_purchase,date_clean,review_length_clean,review_clean,review_clean_tokens,clean_token_count,sentiment
0,########,Review title 1,"Not satisfied, many bugs and issues.",1,user1,80,6,Amazon,zh,Kenya,2.1.4,No,NaT,36,not satisfy many bug issue,"[not, satisfy, many, bug, issue]",5,Negative
1,########,Review title 2,Amazing quality and user-friendly interface.,5,user2,180,5,Flipkart,zh,France,1.2.3,No,NaT,44,amaze quality user friendly interface,"[amaze, quality, user, friendly, interface]",5,Positive
2,########,Review title 3,"Terrible experience, needs major improvements.",2,user3,154,5,Flipkart,pt,USA,1.2.3,No,NaT,46,terrible experience need major improvement,"[terrible, experience, need, major, improvement]",5,Negative
3,########,Review title 4,Poor performance and not user-friendly.,1,user4,96,5,Amazon,es,Qatar,2.1.4,Yes,NaT,39,poor performance not user friendly,"[poor, performance, not, user, friendly]",5,Negative
4,########,Review title 5,"Not satisfied, many bugs and issues.",2,user5,139,6,Website,ar,Kenya,2.1.4,No,NaT,36,not satisfy many bug issue,"[not, satisfy, many, bug, issue]",5,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,2024-07-05 00:00:00,Review title 496,"Waste of time, does not meet expectations.",2,user496,81,7,Flipkart,ja,Qatar,1.2.3,No,2024-07-05,42,waste time not meet expectation,"[waste, time, not, meet, expectation]",5,Negative
496,########,Review title 497,"Great experience, smooth performance and usefu...",5,user497,48,7,Google Play,fr,Nepal,2.1.4,No,NaT,57,great experience smooth performance useful fea...,"[great, experience, smooth, performance, usefu...",6,Positive
497,########,Review title 498,"Terrible experience, needs major improvements.",2,user498,16,5,Google Play,de,Poland,2.1.4,Yes,NaT,46,terrible experience need major improvement,"[terrible, experience, need, major, improvement]",5,Negative
498,########,Review title 499,"Highly satisfied, the app works exactly as exp...",4,user499,68,8,Google Play,zh,Nepal,1.2.3,Yes,NaT,52,highly satisfy app work exactly expect,"[highly, satisfy, app, work, exactly, expect]",6,Positive


In [13]:
df.drop(columns=['title', 'date', 'username'], inplace=True)

In [19]:
df.drop(columns=['review_length', 'review_clean_tokens'], inplace=True)

In [20]:
df

,review,rating,helpful_votes,platform,language,location,version,verified_purchase,date_clean,review_length_clean,review_clean,clean_token_count,sentiment
0,"Not satisfied, many bugs and issues.",1,80,Amazon,zh,Kenya,2.1.4,No,NaT,36,not satisfy many bug issue,5,Negative
1,Amazing quality and user-friendly interface.,5,180,Flipkart,zh,France,1.2.3,No,NaT,44,amaze quality user friendly interface,5,Positive
2,"Terrible experience, needs major improvements.",2,154,Flipkart,pt,USA,1.2.3,No,NaT,46,terrible experience need major improvement,5,Negative
3,Poor performance and not user-friendly.,1,96,Amazon,es,Qatar,2.1.4,Yes,NaT,39,poor performance not user friendly,5,Negative
4,"Not satisfied, many bugs and issues.",2,139,Website,ar,Kenya,2.1.4,No,NaT,36,not satisfy many bug issue,5,Negative
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,"Waste of time, does not meet expectations.",2,81,Flipkart,ja,Qatar,1.2.3,No,2024-07-05,42,waste time not meet expectation,5,Negative
496,"Great experience, smooth performance and usefu...",5,48,Google Play,fr,Nepal,2.1.4,No,NaT,57,great experience smooth performance useful fea...,6,Positive
497,"Terrible experience, needs major improvements.",2,16,Google Play,de,Poland,2.1.4,Yes,NaT,46,terrible experience need major improvement,5,Negative
498,"Highly satisfied, the app works exactly as exp...",4,68,Google Play,zh,Nepal,1.2.3,Yes,NaT,52,highly satisfy app work exactly expect,6,Positive


Save model

In [21]:
df.to_csv('chatgpt_reviews_clean.csv', index=False)

next in 2_eda.ipynb